# ds004752 V5.5 Colab Gate 0 Quickstart

This notebook is an orchestration layer only. All project logic stays in `src/cli.py` and package modules so runs are testable and reproducible.

Use this notebook to mount Google Drive, clone the GitHub repo, prepare the dataset working tree, run tests, and execute Gate 0 metadata audit.

## 1. Mount Drive

Artifacts, cache, checkpoints, and optional materialized data live under `/content/drive/MyDrive/eeg-ds004752`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update the project repo

In [ ]:
# Clone hoặc pull repo private một cách an toàn.
# - Nếu repo chưa tồn tại: hỏi GitHub token bằng getpass, không in token ra output.
# - Nếu repo đã tồn tại: thử git pull thường trước; nếu fail do auth thì dùng token.

from pathlib import Path
import base64
import getpass
import os
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    extra_header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={extra_header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    pull_result = subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)
    if pull_result.returncode != 0:
        extra_header = git_auth_header()
        run(['git', '-c', f'http.extraHeader={extra_header}', 'pull', '--ff-only'], cwd=REPO_DIR)

In [ ]:
%cd /content/eeg-ds004752
!bash bootstrap/install_runtime.sh
!python bootstrap/colab_quickstart.py

## 3. Prepare dataset metadata or payload

Start with `metadata` to create the DataLad working tree without downloading all EDF/MAT payloads. Use `sample` for one subject/session payload, or `all` only when Drive has enough space and you are ready for signal-level audit.

In [ ]:
!bash bootstrap/get_data_colab.sh /content/drive/MyDrive/eeg-ds004752/data metadata

Optional sample payload download for signal-level experiments:

In [ ]:
# !bash bootstrap/get_data_colab.sh /content/drive/MyDrive/eeg-ds004752/data sample

## 4. Run tests and smoke check

In [ ]:
!python -m unittest discover -s tests
!python -m src.cli smoke --profile t4_safe --config configs/data/snapshot_colab.yaml

## 5. Run Gate 0 audit to Drive

In [ ]:
!python -m src.cli audit --profile t4_safe --config configs/data/snapshot_colab.yaml

## 6. Inspect the latest manifest

In [ ]:
import json
from pathlib import Path

latest = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/latest.txt')
run_dir = Path(latest.read_text().strip())
manifest = json.loads((run_dir / 'manifest.json').read_text())
{
    'run_dir': str(run_dir),
    'subjects': manifest['subjects']['n_subjects'],
    'sessions': manifest['subjects']['n_sessions'],
    'eeg_trials': manifest['events_audit']['eeg_trials_total'],
    'ieeg_trials': manifest['events_audit']['ieeg_trials_total'],
    'blockers': manifest['gate0_blockers'],
}

## 7. Guard check

This command should fail until Gate 2.5 preregistration is locked. A failure here is expected and confirms that real-data phases are guarded.

In [ ]:
!python -m src.cli phase1_real --profile a100_fast --config configs/prereg/prereg_bundle.json